Wudu stuff


In [56]:
import io
from pathlib import Path
import pandas as pd

DATA = Path("data/Einzelteil")

FILES = {
    "t01": ("Einzelteil_T01.txt", b" | | ", b" "),
    "t02": ("Einzelteil_T02.txt", b"  ", b"\t"),
    "t03": ("Einzelteil_T03.txt", b"|", b"\x0b"),
    "t04": ("Einzelteil_T04.csv", b";", b"\n"),
    "t05": ("Einzelteil_T05.csv", b",", b"\n"),
}

# Bytes, die beim Teilimport vom Dateianfang gelesen werden.
# Muss groß genug sein, damit nrows+1 vollstaendige Zeilen enthalten sind.
CHUNK_BYTES = 2_000_000


def load(key: str, nrows: int | None = 1000) -> pd.DataFrame:
    filename, field_sep, row_sep = FILES[key]
    path = DATA / filename

    # Byte vodoo (replacing field- and row- seperators)
    if nrows is None:
        raw = path.read_bytes()
        raw = raw.replace(field_sep, b"\x01").replace(row_sep, b"\n")
    else:
        with open(path, "rb") as f:
            raw = f.read(CHUNK_BYTES)
        raw = raw.replace(field_sep, b"\x01").replace(row_sep, b"\n")
        lines = raw.split(b"\n")
        if len(lines) < nrows + 2 and path.stat().st_size > CHUNK_BYTES:
            raise ValueError(
                f"{filename}: {CHUNK_BYTES} Bytes reichen fuer {nrows} Zeilen nicht aus, "
                "CHUNK_BYTES erhoehen"
            )
        # letzte, evtl. abgeschnittene Zeile faellt raus
        raw = b"\n".join(lines[: nrows + 1])

    df = pd.read_csv(io.BytesIO(raw), sep="\x01", na_values=["NA"], dtype=str)

    # Combine same columns (.x, .y)
    for col in set(c.removesuffix(".x").removesuffix(".y") for c in df.columns):
        if col + ".x" in df.columns and col + ".y" in df.columns:
            df[col] = df[col + ".x"].combine_first(df[col + ".y"])

    # Standartize id and time columns
    part = key.upper()
    df["Part_ID"] = df.get(f"ID_{part}", df.get("Part_ID"))

    if "Produktionsdatum" not in df.columns and "Produktionsdatum_Origin_01011970" in df.columns:
        df["Produktionsdatum"] = pd.to_datetime("1970-01-01") + pd.to_timedelta(
            df["Produktionsdatum_Origin_01011970"].astype(float), unit="D"
        )

    # Convert datatypes
    df["Produktionsdatum"] = pd.to_datetime(df["Produktionsdatum"])
    df["Fehlerhaft_Datum"] = pd.to_datetime(df["Fehlerhaft_Datum"])
    df["Fehlerhaft_Fahrleistung"] = df["Fehlerhaft_Fahrleistung"].str.replace(",", ".", regex=False).astype(float)

    int_cols = ["Herstellernummer", "Werksnummer", "Fehlerhaft"]
    df[int_cols] = df[int_cols].astype("Int64")

    columns = [
        "Part_ID",
        "Herstellernummer",
        "Werksnummer",
        "Produktionsdatum",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ]

    return df[columns].reset_index(drop=True)

Importing Einzelteile files

In [57]:
# Loading the part files by using our helper function
t01 = load("t01")
t02 = load("t02")
t03 = load("t03")
t04 = load("t04")
t05 = load("t05")




t01 = t01[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t02 = t02[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t03 = t03[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t04 = t04[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t05 = t05[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
einzelteile = [t01, t02, t03, t04, t05]

#Part type erstellen
t01["Part_Type"] = "T01"
t02["Part_Type"] = "T02"
t03["Part_Type"] = "T03"
t04["Part_Type"] = "T04"
t05["Part_Type"] = "T05"

einzelteile_zusammen = pd.concat([t01, t02, t03, t04, t05],axis=0, ignore_index=True)
part_type_column = einzelteile_zusammen.pop("Part_Type")
einzelteile_zusammen.insert(0, "Part_Type",part_type_column)
display(t01.head(), einzelteile_zusammen.tail(), t04.head())

,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Part_Type
0,1-201-2011-247,201,2008-11-07,0,T01
1,1-201-2011-429,201,2008-11-07,0,T01
2,1-201-2011-363,201,2008-11-07,1,T01
3,1-201-2011-30,201,2008-11-07,0,T01
4,1-201-2011-72,201,2008-11-07,1,T01


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft
4995,T05,5-201-2012-940,201,2008-11-11,1
4996,T05,5-201-2012-356,201,2008-11-08,1
4997,T05,5-201-2012-748,201,2008-11-10,0
4998,T05,5-201-2012-567,201,2008-11-09,1
4999,T05,5-201-2012-674,201,2008-11-10,0


,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Part_Type
0,4-204-2043-113,204,2008-11-07,0,T04
1,4-202-2023-18,202,2008-11-07,0,T04
2,4-204-2043-98,204,2008-11-07,0,T04
3,4-202-2023-51,202,2008-11-07,0,T04
4,4-204-2043-169,204,2008-11-07,0,T04


importing komponente data

In [ ]:
file_paths = [
    'data/Komponente/Bestandteile_Komponente_K1BE1.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI1.csv',
    'data/Komponente/Bestandteile_Komponente_K1BE2.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI2.csv'
]

engine_dfs = []

# for path in file_paths:
#     df = pd.read_csv(path, sep=';').drop(columns=['Unnamed: 0'])
#     engine_dfs.append(df)


komponente_k1be1 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1BE1.csv", sep=';').drop(columns=['Unnamed: 0'])

display(komponente_k1be1.head())


,ID_T1,ID_T2,ID_T3,ID_T4,ID_K1BE1
0,1-201-2011-45,2-201-2011-161,3-202-2023-14,4-202-2023-20,K1BE1-101-1011-1
1,1-201-2011-429,2-201-2011-239,3-202-2023-16,4-202-2023-51,K1BE1-101-1011-2
2,1-201-2011-399,2-201-2011-220,3-202-2023-46,4-202-2023-93,K1BE1-101-1011-3
3,1-201-2011-335,2-202-2022-463,3-202-2023-149,4-204-2042-18,K1BE1-101-1011-4
4,1-204-2044-188,2-202-2022-675,3-202-2023-152,4-204-2042-40,K1BE1-101-1011-5


Melting the data

In [59]:
melted = komponente_k1be1.melt(id_vars="ID_K1BE1", var_name="Part_Type", value_name="Part_ID")
melted = melted.sort_values(["ID_K1BE1", "Part_Type"])
melted = melted.drop(columns=["Part_Type"])
display(melted.head())

,ID_K1BE1,Part_ID
0,K1BE1-101-1011-1,1-201-2011-45
1192630,K1BE1-101-1011-1,2-201-2011-161
2385260,K1BE1-101-1011-1,3-202-2023-14
3577890,K1BE1-101-1011-1,4-202-2023-20
9,K1BE1-101-1011-10,1-201-2011-37


In [60]:
display(einzelteile_zusammen.head(), melted.head())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft
0,T01,1-201-2011-247,201,2008-11-07,0
1,T01,1-201-2011-429,201,2008-11-07,0
2,T01,1-201-2011-363,201,2008-11-07,1
3,T01,1-201-2011-30,201,2008-11-07,0
4,T01,1-201-2011-72,201,2008-11-07,1


,ID_K1BE1,Part_ID
0,K1BE1-101-1011-1,1-201-2011-45
1192630,K1BE1-101-1011-1,2-201-2011-161
2385260,K1BE1-101-1011-1,3-202-2023-14
3577890,K1BE1-101-1011-1,4-202-2023-20
9,K1BE1-101-1011-10,1-201-2011-37


Merging the df

In [62]:
merged2 = pd.merge(einzelteile_zusammen, melted, how='outer', on="Part_ID")
merged2 = merged2.sort_values(["ID_K1BE1", "Part_Type"])
display(merged2.head())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1BE1
250376,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1
1218401,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1
2764448,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1
3689809,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1
217342,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10


Checking if motor is in OM1

In [ ]:
# Importing OEM1 data
oem11 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")
oem12 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")

# Combining the OEM1 Types
oem_combined = pd.concat([oem11, oem12], ignore_index=True)

display(oem_combined.head())


,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor,ID_Fahrzeug
0,K4-112-1121-3,K3SG1-105-1051-32,K2LE1-109-1091-2,K1BE1-101-1011-7,11-1-11-1
1,K4-112-1121-4,K3SG1-105-1051-141,K2ST1-109-1092-5,K1BE1-101-1011-12,11-1-11-2
2,K4-112-1121-7,K3SG1-105-1051-106,K2ST1-109-1092-57,K1BE1-101-1011-38,11-1-11-3
3,K4-112-1121-9,K3SG1-105-1051-21,K2ST1-109-1092-91,K1BE1-101-1011-97,11-1-11-4
4,K4-112-1121-11,K3SG1-105-1051-59,K2ST1-109-1092-4,K1BE1-101-1011-65,11-1-11-5
